In [50]:
import numpy as np
import networkx as nx
import re
from argparse import Namespace
from itertools import combinations
from collections import Counter


from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Parameter, ParameterVector
from qiskit.transpiler import Layout
from qiskit.transpiler.passes import LayoutTransformation
from qiskit.converters import dag_to_circuit, circuit_to_dag

from qiskit_aer import AerSimulator
from qiskit_aer.backends.backendconfiguration import AerBackendConfiguration
from qiskit_aer.primitives import SamplerV2 as Sampler

from qiskit_qaoa.hubo.graph_to_hubo_hamiltonian import graph_to_hubo_hamiltonian
from qiskit_qaoa.utils.gfa_utils import gfa_file_to_graph
from qiskit_qaoa.utils.transpiler_passes import ExtendedSwapStrategy
from qiskit_qaoa.utils.layout_utils import swap_between_circuit_layouts
from qiskit_qaoa.utils.string_utils import evaluate_sparse_pauli_samples


In [51]:
scores = {i: np.exp(-1j*i) for i in np.linspace(-100,100, 200+1)}
def get_score(x: complex):
    for i in np.linspace(-100,100, 200+1):
        if np.abs(scores[i]- x) < 1e-5:
            return i 
    return x

In [75]:
rng = np.random.default_rng(seed=1)

args = Namespace(**dict(
    filename='test_N4_W5',
    extra=1,
    fraction_four=0.0,
    fraction_six=1.0,
    timeout=5,
    copy_numbers=[2,1,1,1],
    coupling_map='grid',
    reps=4,
    method='Powell',
    shots=1000,
    init='random',
    alpha=0.05
))


p = args.reps

In [76]:
    
filepath = f'/nfs/users/nfs_j/jc59/quantumwork/pangenome/data/{args.filename}.gfa'
graph, n, V, T = gfa_file_to_graph(filepath, args.copy_numbers)
num_qubits = n*T
print(f'Virtual qubits: {num_qubits}')


if args.coupling_map == 'line':
    extended_swap_strat = ExtendedSwapStrategy.from_line(list(range(num_qubits)), num_swap_layers=1000)
elif args.coupling_map == 'grid':
    extended_swap_strat = ExtendedSwapStrategy.from_grid(n, T)
else:
    raise Exception('Invalid coupling map type')

num_physical_qubits = extended_swap_strat._num_vertices
coupling_map = extended_swap_strat._coupling_map
    
coupling_map_edge = list(coupling_map)
physical_qubits = list(coupling_map.physical_qubits)
dual_coupling_map = nx.Graph()

for qubit in physical_qubits:
    edges = [edge for edge in coupling_map_edge if edge[0]==qubit]
    for edge1, edge2 in combinations(edges, 2):
        dual_coupling_map.add_edge(tuple(sorted(edge1)), tuple(sorted(edge2)))
edge_colouring = nx.greedy_color(dual_coupling_map, interchange=True)



print(f'Physical qubits: {num_physical_qubits}')

basis_gates=["sx", "x", "rz", "rzz", "cz", "id", "cx", "swap", "h"]

backend_options = dict(
    method='statevector',
    device='CPU',
    precision='single',
    basis_gates=basis_gates
)


config = AerSimulator._DEFAULT_CONFIGURATION
config["n_qubits"] = num_physical_qubits
config["basis_gates"] = basis_gates
config = AerBackendConfiguration.from_dict(config)
backend = AerSimulator(configuration=config, **backend_options)
backend.set_option("n_qubits", num_physical_qubits)
sampler = Sampler().from_backend(backend)
print(backend.configuration().to_dict()["n_qubits"])

full_hamiltonian = args.reps * graph_to_hubo_hamiltonian(graph, n, T, lamda=10 / args.reps, fraction_terms=1.0)

Virtual qubits: 20
Physical qubits: 20
20
Keeping constraints at times: [1 3 0 2]


In [77]:
import pickle
with open(f'./per_layer_results.{args.filename}.reps{args.reps}.pkl', 'rb') as f:
    data = pickle.load(f)
hamiltonians = data["hamiltonians"]
swap_depths = data["swap_depths"]
edge_maps = data["edge_maps"]
compiled_circuits = data["compiled_circuits"]

In [78]:
donor_qc = QuantumCircuit(num_qubits)
qaoa_circuit = QuantumCircuit(0, num_qubits)
qaoa_circuit.add_register([donor_qc.qubits[edge_maps[0][i]] for i in range(num_qubits)])

mixer_layer = QuantumCircuit(num_qubits)
beta = Parameter("β")
mixer_layer.rx(2 * beta, range(num_qubits))

gammas = ParameterVector("γ", args.reps)
betas = ParameterVector("β", args.reps)

# for i in range(num_qubits):
#     qaoa_circuit.h(i)

for i in [7,8]:
    qaoa_circuit.x(edge_maps[0][i])


for layer in range(0, args.reps):
    swap_circuit = swap_between_circuit_layouts(layer-1, compiled_circuits, edge_maps, extended_swap_strat._coupling_map)
    qaoa_circuit.compose(swap_circuit, range(num_qubits), inplace=True)

    bind_dict = {compiled_circuits[layer].parameters[0]: gammas[layer]}
    bound_cost_layer = compiled_circuits[layer].assign_parameters(bind_dict)
    qaoa_circuit.compose(bound_cost_layer, range(num_qubits), inplace=True)
   
    bind_dict = {mixer_layer.parameters[0]: betas[layer]}
    bound_mixer_layer = mixer_layer.assign_parameters(bind_dict)
    qaoa_circuit.compose(bound_mixer_layer, range(num_qubits), inplace=True)


layer = args.reps - 1
final_layout = Layout({i: donor_qc.qubits[edge_maps[layer][i]] for i in range(num_qubits)})
for instruction in compiled_circuits[layer].data:
    if instruction.operation.name == 'swap':
        qubits_str = str(instruction.qubits)
        matches = re.findall('index=([0-9]+)', qubits_str)
        if len(matches) == 2:
            final_layout.swap(int(matches[0]), int(matches[1]))
        else:
            raise Exception('Did not find 2 swap indices')


qaoa_circuit.save_statevector(label='before_swaps')

# to_layout = Layout({i: donor_qc.qubits[i] for i in range(num_qubits)})
# transformation_pass = LayoutTransformation(None,final_layout, to_layout)
# swap_qc = QuantumCircuit(num_qubits)
# swap_qc = dag_to_circuit(transformation_pass.run(circuit_to_dag(swap_qc)))
# qaoa_circuit.compose(swap_qc, range(num_qubits), inplace=True)
qaoa_circuit.save_statevector()
for physical, virtual in (
    final_layout.get_physical_bits().items()
):
    qaoa_circuit.measure(physical, donor_qc.find_bit(virtual).index)

In [79]:
params = [0] * (args.reps - 1) + [0] + [1] * args.reps
tqaoa_for_backend = transpile(qaoa_circuit, optimization_level=0, backend=backend)
print(tqaoa_for_backend.parameters)
bqaoa_for_backend = tqaoa_for_backend.assign_parameters({param: params[idx] for idx, param in enumerate(tqaoa_for_backend.parameters)})
res_custom = backend.run(bqaoa_for_backend,shots=1000).result()

ParameterView([ParameterVectorElement(β[0]), ParameterVectorElement(β[1]), ParameterVectorElement(β[2]), ParameterVectorElement(β[3]), ParameterVectorElement(γ[0]), ParameterVectorElement(γ[1]), ParameterVectorElement(γ[2]), ParameterVectorElement(γ[3])])


In [80]:
data = res_custom.results[0].data
custom_sv = np.asarray(data.statevector)
before_swaps = np.asarray(data.before_swaps)

In [81]:
res_custom.get_counts()

{'00000000000100000010': 1000}

In [59]:
reorder_sv = before_swaps.reshape([2]*num_physical_qubits).transpose(
    [donor_qc.find_bit(final_layout.get_physical_bits()[physical]).index for physical in sorted(list(final_layout.get_physical_bits().keys()))]
).reshape((2**num_qubits,))
# reorder_sv = before_swaps.reshape([2]*num_physical_qubits).transpose().transpose(
#     [0,1,8,3,4,5,6,7,2]
# ).transpose().reshape((2**num_qubits,))
# np.transpose(
#     before_swaps.reshape([2]*num_physical_qubits), 
#     [donor_qc.find_bit(final_layout.get_physical_bits()[physical]).index for physical in sorted(list(final_layout.get_physical_bits().keys()))]
# ).reshape((2**num_qubits,))

In [60]:
# # for idx in range(0, 20):
# for idx in range(len(custom_sv)):
#     x = custom_sv[idx]
#     if not get_score(x * 2**(num_physical_qubits/2)) == evaluate_sparse_pauli_samples([np.binary_repr(idx, num_physical_qubits)], full_hamiltonian)[0]:
#         print(
#             np.binary_repr(idx, num_physical_qubits),
#             get_score(x * 2**(num_physical_qubits/2)),
#             evaluate_sparse_pauli_samples([np.binary_repr(idx, num_physical_qubits)], full_hamiltonian)[0]
#         )
#         print()

In [61]:
# # for idx in range(0, 20):
# for idx in range(len(reorder_sv)):
#     x = reorder_sv[idx]
#     if not get_score(x * 2**(num_physical_qubits/2)) == evaluate_sparse_pauli_samples([np.binary_repr(idx, num_physical_qubits)], full_hamiltonian)[0]:
#         print(
#             np.binary_repr(idx, num_physical_qubits),
#             get_score(x * 2**(num_physical_qubits/2)),
#             evaluate_sparse_pauli_samples([np.binary_repr(idx, num_physical_qubits)], full_hamiltonian)[0]
#         )
#         print()

In [62]:
len(np.nonzero(np.abs(custom_sv-reorder_sv) > 1e-6)[0])

2

In [63]:
# donor_qc = QuantumCircuit(3)
# test = QuantumCircuit(0, 3)
# edge_maps = {
#     0: {0:1,1:2,2:0},
#     1: {0:2,1:1,2:0}
# }
# test.add_register([donor_qc.qubits[edge_maps[0][i]] for i in range(3)])
# test.h(edge_maps[0][0])
# # test.rx(np.pi/4, 1)
# test.x(edge_maps[0][2])

# test.barrier()
# test.swap(1,2)
# test.swap(2,0)
# from_layout = Layout({i: donor_qc.qubits[edge_maps[0][i]] for i in range(3)})
# from_layout.swap(1, 2)
# from_layout.swap(2, 0)

# to_layout = Layout({i: donor_qc.qubits[edge_maps[1][i]] for i in range(3)})
# transformation_pass = LayoutTransformation(coupling_map, from_layout, to_layout)
# swap_qc = QuantumCircuit(3)
# swap_qc = dag_to_circuit(transformation_pass.run(circuit_to_dag(swap_qc)))
# test.barrier()

# test.compose(swap_qc, range(3), inplace=True)

# test.barrier()

# test.swap(1,0)
# test.swap(0,2)
# test.barrier()

# final_layout = Layout({i: donor_qc.qubits[edge_maps[1][i]] for i in range(3)})
# final_layout.swap(1, 0)
# final_layout.swap(0, 2)

# # follow this was a small swap network and layout transformation ciurcuits and measurmeents as defined above and see what the outcome strings look like
# test.save_statevector()
# for physical, virtual in (
#     final_layout.get_physical_bits().items()
# ):
#     test.measure(physical, donor_qc.find_bit(virtual).index)



In [64]:
# test.draw(fold=-1)

In [65]:
# test_res = backend.run(test).result()
# data = test_res.results[0].data
# test_sv = np.asarray(data.statevector)
# # test_sv = test_sv.reshape([2,2,2])

In [66]:
# test_res.get_counts()
# Hadamard virtual 0, line 1, gets swapped to line 1, and measured onto clbit 2, which is the left-most
# |0> virtual 1, line 2 gets swapped to line 0, and measured onto clbit 0, which is the right-most
# |1> virtual 2, line 0 gets swapped to line 2, and measured onto clbit 1, which is the middle